# Reto 01 — Perfiles de gestión municipal de residuos (GIRS 2024)

**Pontificia Universidad Católica del Ecuador · Ciencia de Datos · Aprendizaje Automático Inicial**  
**Equipo:** Isabel Paguay · Mateo Vera · Martín Andrade  
**Docente:** Andrés Merino · **Edición de datos:** INEC GIRS 2024

### Pregunta de investigación
¿Qué perfiles exploratorios de gestión de residuos aparecen en los municipios del Ecuador y cómo pueden ayudar a formular preguntas de mejora?

**Objetivo del notebook final (Hito 4).** Partir del **CSV ya preparado** del Hito 2, auditar sus controles, construir las matrices de 215 y 204 municipios, comparar K-Means y Ward para $k=2,\ldots,6$, interpretar cinco perfiles, comprobar su sensibilidad y representar los resultados con PCA. Las métricas se calculan en **cinco dimensiones**; PCA solamente se usa para visualizar.

> **Alcance:** los datos son declaraciones municipales de 2024. Los grupos no son un ranking de calidad, no demuestran causalidad y no autorizan por sí solos asignar recursos. Los 221 municipios se conservan en la auditoría, aunque seis no puedan integrarse en la matriz completa.

## 1. Procedencia, preparación anterior y forma de ejecución

- **Fuente:** INEC, *Estadística de Información Ambiental Económica en GAD Municipales: Gestión de Residuos Sólidos, edición 2024*. Base oficial de referencia: 221 registros, 213 columnas, `BDD_GIRS_2024_VF.csv` (original con separador `;`, decimales con coma y codificación Windows-1252).
- **Entrada real de este Colab:** `02_hito2/datos/base_preparada_221.csv`, una **tabla ya limpiada y documentada** con 22 columnas derivadas de un extracto de 41 campos. **No** representa ni reemplaza el CSV oficial de 213 columnas.
- **Repositorio:** [2007Mateo/Aprendizaje_Automatico_Reto01](https://github.com/2007Mateo/Aprendizaje_Automatico_Reto01). Se usa un **commit fijo** para facilitar la reproducción.
- **Fuente del diccionario:** [INEC/ANDA GIRS 2024](https://anda.inec.gob.ec/anda5/index.php/catalog/1294/data-dictionary). Fecha de consulta de los materiales del equipo: 20 de noviembre de 2026; fecha original de descarga del INEC no consta en la evidencia transferida.

Ejecutar de arriba hacia abajo en **Google Colab**. Por defecto se carga automáticamente el CSV preparado desde GitHub. Si el docente entrega una copia local, cambiar `MODO_CARGA` a `"subir"`. **Nunca** completar o reconstruir `CantonId` a partir de su apariencia numérica: se conserva como texto tal como llegó del archivo preparado.

In [ ]:
from pathlib import Path
from itertools import combinations
from datetime import datetime, timezone
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn, scipy, json, os, sys, zipfile, hashlib
from io import BytesIO
from urllib.request import urlopen
from IPython.display import display

# Usar "repositorio" para la entrega reproducible, o "subir" para cargar el CSV manualmente.
MODO_CARGA = "repositorio"
COMMIT_FIJO = "5783cc0c1f65d492ed6e953e185091c378ed8c68"
RUTA_CSV = "02_hito2/datos/base_preparada_221.csv"
URL_CSV = f"https://raw.githubusercontent.com/2007Mateo/Aprendizaje_Automatico_Reto01/{COMMIT_FIJO}/{RUTA_CSV}"
CARPETA_SALIDA = Path("resultados_reto01_final")
CARPETA_SALIDA.mkdir(exist_ok=True)
SEMILLAS = (11, 23, 37, 52, 97)
VALORES_K = (2, 3, 4, 5, 6)
K_ELEGIDO, SEMILLA_ELEGIDA, N_INICIALIZACIONES = 5, 11, 30
ATRIBUTOS = ["CSB_cobertura", "DIF_proporcion", "ORG_actividad", "RI_log_p95", "BOTADERO_indicador"]

print("Python:", sys.version.split()[0], "| NumPy:", np.__version__,
      "| scikit-learn:", sklearn.__version__, "| SciPy:", scipy.__version__)
if sklearn.__version__ != "1.8.0":
    print("AVISO: el Hito 3 se verificó con scikit-learn 1.8.0; comprobar los controles finales.")

In [ ]:
# GIRS_CSV_LOCAL permite probar una copia preparada sin usar red (opcional).
archivo_local = os.environ.get("GIRS_CSV_LOCAL")
if archivo_local:
    origen = archivo_local
elif MODO_CARGA == "subir":
    from google.colab import files
    archivos = files.upload()
    if len(archivos) != 1:
        raise ValueError("Subir exactamente un CSV preparado de 221 filas")
    origen = next(iter(archivos))
else:
    origen = URL_CSV

# Verificar los bytes de entrada y preservar la conversión decimal exacta.
# Con la lectura decimal predeterminada, una diferencia inferior a 1e-15
# puede cambiar el resultado de la semilla 52 y el ARI de estabilidad.
SHA256_CSV_PREPARADO = "6dbd7aa53b4f93ed5c09b2037275abc2272250221eee52ebd1b892b48ffdcb59"
if str(origen).startswith(("https://", "http://")):
    with urlopen(str(origen), timeout=45) as respuesta:
        contenido_csv = respuesta.read()
else:
    contenido_csv = Path(origen).read_bytes()
sha256_real = hashlib.sha256(contenido_csv).hexdigest()
coincide_csv_institucional_preparado = sha256_real == SHA256_CSV_PREPARADO
print("SHA256 del CSV recibido:", sha256_real)
if MODO_CARGA == "repositorio" and not archivo_local:
    if not coincide_csv_institucional_preparado:
        raise AssertionError("El CSV del commit fijado NO coincide con el manifiesto del Hito 2. Detener y auditar.")
    print("Integridad del CSV preparado del repositorio: CONFIRMADA")
else:
    print("Archivo proporcionado externamente; coincide con el manifiesto:",coincide_csv_institucional_preparado)

base = pd.read_csv(BytesIO(contenido_csv), encoding="utf-8-sig",
                   dtype={"CantonId":"string", "Provin":"string"},
                   float_precision="round_trip")
base.columns = [c.strip().replace("\ufeff", "") for c in base.columns]
base["CantonId"] = base["CantonId"].astype("string").str.strip()
print("Origen del CSV preparado:", origen)
print("Dimensiones del CSV preparado:", base.shape)
display(base.head(5))

## 2. Auditoría de la entrada limpia (sin rehacer el Hito 2)

Se conservan **221 municipios únicos**. En la auditoría previa se anotaron **17 incidencias**: cinco coberturas de barrido no calculables, la proporción de diferenciación de **Piñas** por encima de 1 y once registros con recuperación inorgánica cero a pesar de declarar actividad relacionada. Cuatro coberturas de barrido mayores a 1 se preservan sin truncamiento. Los once ceros ambiguos **se conservan** en la muestra principal y se excluyen solo en la muestra conservadora.

In [ ]:
requeridas = {"CantonId","NombreCantón","Provin","Estricto_204","Alternativo_215",
    "CSB_original","CSB_estado","DIF_proporcion","DIF_estado","ORG_indicador",
    "RI_intensidad","RI_estado","RI_log_p95","BOTADERO_indicador",
    "CRR_ton_mes_contexto","RI_kg_mes","Material_recuperado_si"}
faltantes = sorted(requeridas - set(base.columns))
if faltantes:
    raise ValueError(f"Este no es el CSV limpio esperado. Faltan: {faltantes}")
assert base.shape[0] == 221, f"El conjunto esperado contiene 221 municipios: se leyeron {len(base)}"
assert base["CantonId"].notna().all() and base["CantonId"].nunique() == 221, "IDs municipales ausentes o repetidos"
assert all(base["CantonId"].str.fullmatch(r"\d+")), "IDs no numéricos en texto: cotejar con el origen"

numericas = ["Estricto_204", "Alternativo_215", "CSB_original", "DIF_proporcion",
    "ORG_indicador", "RI_intensidad", "RI_log_p95", "BOTADERO_indicador",
    "CRR_ton_mes_contexto", "RI_kg_mes"]
for c in numericas:
    base[c] = pd.to_numeric(base[c], errors="raise")

controles = {
    "municipios_unicos": int(base["CantonId"].nunique()),
    "sin_cobertura_barrido": int(base["CSB_estado"].eq("SIN DENOMINADOR").sum()),
    "diferenciacion_invalida": int(base["DIF_estado"].eq("INVALIDO").sum()),
    "ceros_RI_ambiguos": int(base["RI_estado"].eq("CERO AMBIGUO").sum()),
    "coberturas_CSB_superiores_a_uno": int((base["CSB_original"] > 1).sum()),
    "filas_principal": int(base["Alternativo_215"].eq(1).sum()),
    "filas_conservadora": int(base["Estricto_204"].eq(1).sum())
}
display(pd.DataFrame([controles]).T.rename(columns={0: "Valor observado"}))
assert (controles["sin_cobertura_barrido"], controles["diferenciacion_invalida"],
        controles["ceros_RI_ambiguos"], controles["coberturas_CSB_superiores_a_uno"]) == (5,1,11,4)
assert (controles["filas_principal"], controles["filas_conservadora"]) == (215,204)
print("Auditoría de entrada: OK. Los 17 casos documentados se conservan identificados.")

## 3. Selección de características y matrices de análisis

| Atributo usado por los algoritmos | Fuente en el CSV preparado | Interpretación / unidad | Decisión |
|---|---|---|---|
| `CSB_cobertura` | `CSB_original` | Cobertura de barrido (proporción) | Conservar los cuatro valores mayores a 1, sin recorte. |
| `DIF_proporcion` | `DIF_proporcion` | Fracción de recolección diferenciada | Ya fue calculada con cantidades y denominadores compatibles. |
| `ORG_actividad` | `ORG_indicador` | Sí/no de actividad orgánica declarada | **No** equivale a toneladas tratadas. |
| `RI_log_p95` | `RI_intensidad`, `RI_log_p95` | Intensidad relativa transformada | $r_i=RI_{kg/mes}/(1000\,CRR_{t/mes})$; $\log(1+100r_i)/\log(1+100q_{95})$, con $q_{95}$ fijo calculado de 204 casos. |
| `BOTADERO_indicador` | `BOTADERO_indicador` | Presencia de botadero como disposición principal (0/1) | Bandera declarativa, no escala continua. |

No se incluyen `CantonId`, nombre, provincia ni volumen `CRR` en las **distancias**. Se usa la matriz preparada **sin z-score adicional** en el modelo principal. Todos los vectores municipales se conservan incluso si coinciden exactamente con los de otros municipios.

In [ ]:
principal = base.loc[base["Alternativo_215"] == 1].copy().reset_index(drop=True)
conservadora = base.loc[base["Estricto_204"] == 1].copy().reset_index(drop=True)
for conjunto in (principal, conservadora):
    conjunto["CSB_cobertura"] = conjunto["CSB_original"]
    conjunto["ORG_actividad"] = conjunto["ORG_indicador"]
    assert conjunto[ATRIBUTOS].notna().all().all()
    assert np.isfinite(conjunto[ATRIBUTOS].to_numpy(dtype=float)).all()
X215 = principal[ATRIBUTOS].to_numpy(dtype=float)
X204 = conservadora[ATRIBUTOS].to_numpy(dtype=float)
assert X215.shape == (215,5) and X204.shape == (204,5)
assert set(conservadora.CantonId).issubset(set(principal.CantonId))
assert np.unique(X215,axis=0).shape[0] == 141
assert np.unique(X204,axis=0).shape[0] == 137

q95_rel = np.percentile(conservadora["RI_intensidad"].to_numpy(dtype=float),95)
assert q95_rel > 0
reconstruida = np.log1p(100*principal["RI_intensidad"].to_numpy(dtype=float)) / np.log1p(100*q95_rel)
diferencia_maxima = np.max(np.abs(reconstruida - principal["RI_log_p95"].to_numpy(dtype=float)))
print("q95 relativo fijo (204):", q95_rel, "| desviación reconstruida máxima:", diferencia_maxima)
assert np.allclose(reconstruida, principal["RI_log_p95"], atol=1e-8, rtol=1e-8), "Revisar transformación RI"
print("Matriz principal:", X215.shape, "| Matriz conservadora:", X204.shape)
print("Vectores diferentes por escenario: 141 y 137. Los vectores repetidos no son IDs duplicados.")

In [ ]:
# Revisión exploratoria de distribuciones de los cinco indicadores.
display(principal[ATRIBUTOS].describe().T.round(4))
fig, axs = plt.subplots(2,3,figsize=(14,7))
for ax, col in zip(axs.ravel(),ATRIBUTOS):
    valores = principal[col]
    if col in ("ORG_actividad","BOTADERO_indicador"):
        valores.value_counts().sort_index().reindex([0,1],fill_value=0).plot.bar(ax=ax)
        ax.set_xticklabels(["No (0)","Sí (1)"], rotation=0)
    else:
        ax.hist(valores,bins=18,edgecolor="white")
    ax.set_title(col);ax.set_ylabel("Municipios");ax.grid(axis="y",alpha=.2)
axs.ravel()[-1].axis("off")
fig.suptitle("Distribuciones de la matriz principal (n = 215)")
plt.tight_layout()
fig.savefig(CARPETA_SALIDA/"01_distribuciones.png",dpi=160,bbox_inches="tight")
plt.show()

## 4. Comparación de agrupamientos: K-Means y Ward

Ambos modelos usan **distancia euclidiana** sobre exactamente las **cinco características** anteriores. Ward es un método aglomerativo que minimiza el aumento de variabilidad dentro de los grupos. Para K-Means se prueban cinco semillas (11, 23, 37, 52, 97), cada una con `n_init=30` y `max_iter=500`. Se evalúan todos los valores $k\in\{2,3,4,5,6\}$ en ambos escenarios, con silueta promedio, tamaños y ARI entre semillas.

**Importante:** ARI compara asignaciones sin depender de los números arbitrarios de las etiquetas. La silueta se calcula aquí, **antes** de cualquier PCA.
**Nota de precisión:** los decimales se leen con `float_precision="round_trip"` para no introducir diferencias subyacentes por redondeo en la lectura del CSV. Esta medida es relevante para comprobar las cinco semillas sobre el archivo original.


In [ ]:
def kmeans(X,k,semilla=SEMILLA_ELEGIDA):
    return KMeans(n_clusters=k, random_state=semilla,
                  n_init=N_INICIALIZACIONES,max_iter=500).fit_predict(X)

def ward(X,k):
    return AgglomerativeClustering(n_clusters=k,linkage="ward",metric="euclidean").fit_predict(X)

def silueta(X,etiquetas):
    return float(silhouette_score(X,etiquetas,metric="euclidean"))

def alinear_y_contar(a,b):
    """Número máximo de coincidencias tras una permutación óptima de etiquetas."""
    ua,ub=np.unique(a),np.unique(b)
    tabla=np.array([[np.sum((a==x)&(b==y)) for y in ub] for x in ua],dtype=int)
    filas,columnas=linear_sum_assignment(-tabla)
    return int(tabla[filas,columnas].sum())

resultados, iniciaciones, etiquetas_guardadas = [],[],{}
for nombre,X in (("215",X215),("204",X204)):
    for k in VALORES_K:
        lotes={s:kmeans(X,k,s) for s in SEMILLAS}
        pares=[adjusted_rand_score(lotes[a],lotes[b]) for a,b in combinations(SEMILLAS,2)]
        for s,y in lotes.items():
            iniciaciones.append({"escenario":int(nombre),"k":k,"semilla":s,
                 "silueta_5d":silueta(X,y),"grupo_minimo":int(np.bincount(y).min())})
        ykm=lotes[SEMILLA_ELEGIDA]
        yward=ward(X,k)
        etiquetas_guardadas[(nombre,"K-Means",k)]=ykm
        etiquetas_guardadas[(nombre,"Ward",k)]=yward
        acuerdo=adjusted_rand_score(ykm,yward)
        for metodo,y in (("K-Means",ykm),("Ward",yward)):
            resultados.append({"escenario":int(nombre),"metodo":metodo,"k":k,
                "silueta_5d":silueta(X,y),
                "tamanos":tuple(sorted(np.bincount(y).tolist(),reverse=True)),
                "grupo_minimo":int(np.bincount(y).min()),
                "grupo_maximo":int(np.bincount(y).max()),
                "ARI_KMeans_vs_Ward":float(acuerdo),
                "ARI_semillas_medio":float(np.mean(pares)) if metodo=="K-Means" else np.nan,
                "ARI_semillas_minimo":float(np.min(pares)) if metodo=="K-Means" else np.nan})
comparacion=pd.DataFrame(resultados)
semillas_df=pd.DataFrame(iniciaciones)
assert len(comparacion)==20 and len(semillas_df)==50
print("20 evaluaciones de método × k × escenario y 50 ajustes K-Means entre semillas.")
display(comparacion[comparacion.escenario==215].round(6))

In [ ]:
res_215=comparacion[comparacion.escenario==215]
fig,ax=plt.subplots(figsize=(9,5))
for metodo,estilo in (("K-Means","o-"),("Ward","s-")):
    d=res_215[res_215.metodo==metodo].sort_values("k")
    ax.plot(d.k,d.silueta_5d,estilo,label=metodo,linewidth=2)
ax.axvline(K_ELEGIDO,color="gray",linestyle="--",alpha=.8)
ax.set(xlabel="Número de grupos k",ylabel="Silueta promedio (5 dimensiones)",
       title="Separación de K-Means y Ward · 215 municipios",xticks=list(VALORES_K))
ax.grid(alpha=.2);ax.legend();plt.tight_layout()
fig.savefig(CARPETA_SALIDA/"02_comparacion_siluetas.png",dpi=160,bbox_inches="tight")
plt.show()

print("k=6 tiene mejor silueta, pero su grupo mínimo K-Means tiene 10 municipios.")
print("Se propone k=5 por equilibrio entre separación, tamaño e interpretación; k=4 y 6 siguen siendo alternativas.")

## 5. Solución descriptiva de cinco perfiles

Se selecciona **K-Means con $k=5$, semilla 11**, sin concluir que cinco sea universalmente óptimo. Para evitar confundir IDs numéricos con niveles de desempeño, los nombres se asignan a partir de los patrones y se muestran en un orden convencional G0–G4. No se pueden comparar directamente los IDs de K-Means con los de Ward: se comparan tamaños **ordenados**, nunca barras enfrentadas bajo la misma etiqueta arbitraria.

In [ ]:
YKM_NATIVE=etiquetas_guardadas[("215","K-Means",5)]
YWARD=etiquetas_guardadas[("215","Ward",5)]
PERFILES_ORDEN={
  "Sin diferenciación ni actividad orgánica":0,
  "Diferenciada y actividad orgánica":1,
  "Botadero predominante, poca diferenciación":2,
  "Diferenciada sin actividad orgánica":3,
  "Actividad orgánica, poca diferenciación":4
}

def clasificar_perfil(m):
    if m[4]>.5: return "Botadero predominante, poca diferenciación"
    if m[2]>.5 and m[1]>.5: return "Diferenciada y actividad orgánica"
    if m[2]>.5: return "Actividad orgánica, poca diferenciación"
    if m[1]>.5: return "Diferenciada sin actividad orgánica"
    return "Sin diferenciación ni actividad orgánica"

asignacion_semantica={}
for g in np.unique(YKM_NATIVE):
    nombre=clasificar_perfil(X215[YKM_NATIVE==g].mean(axis=0))
    if nombre in asignacion_semantica.values():
        raise AssertionError("Dos grupos tienen el mismo nombre; revisar solución y versión del algoritmo")
    asignacion_semantica[int(g)]=nombre
assert set(asignacion_semantica.values())==set(PERFILES_ORDEN)
YKM=np.array([PERFILES_ORDEN[asignacion_semantica[int(g)]] for g in YKM_NATIVE],dtype=int)

silueta_km=silueta(X215,YKM)
silueta_ward=silueta(X215,YWARD)
ari_metodos=adjusted_rand_score(YKM,YWARD)
silueta_individual=silhouette_samples(X215,YKM,metric="euclidean")
print(f"K-Means k=5: silueta {silueta_km:.6f}; Ward: {silueta_ward:.6f}; ARI: {ari_metodos:.6f}")
print("Tamaños K-Means G0..G4:",np.bincount(YKM).tolist())
print("Tamaños Ward, de mayor a menor:",sorted(np.bincount(YWARD).tolist(),reverse=True))
assert sorted(np.bincount(YKM).tolist())==[18,19,29,42,107]

fig,ax=plt.subplots(figsize=(8,4))
orden_km=sorted(np.bincount(YKM),reverse=True)
orden_ward=sorted(np.bincount(YWARD),reverse=True)
x=np.arange(5)
ax.bar(x-.17,orden_km,width=.34,label="K-Means")
ax.bar(x+.17,orden_ward,width=.34,label="Ward")
ax.set(xticks=x,xticklabels=["1.º mayor","2.º","3.º","4.º","5.º menor"],
       xlabel="Posición por tamaño (NO equivalente a etiqueta)",ylabel="Municipios",
       title="Tamaños de grupos comparados correctamente · k=5")
ax.legend();ax.grid(axis="y",alpha=.2);plt.tight_layout()
fig.savefig(CARPETA_SALIDA/"03_tamanos_ordenados.png",dpi=160,bbox_inches="tight")
plt.show()

In [ ]:
filas_perfiles,filas_representantes=[],[]
for g in range(5):
    mask=YKM==g
    d=principal.loc[mask]
    xs=X215[mask]
    centroide=xs.mean(axis=0)
    filas_perfiles.append({
      "grupo":f"G{g}","perfil":next(nombre for nombre,num in PERFILES_ORDEN.items() if num==g),
      "municipios":len(d),"barrido_medio_pct":100*xs[:,0].mean(),
      "diferenciada_media_pct":100*xs[:,1].mean(),
      "organicos_pct_municipios":100*xs[:,2].mean(),
      "botadero_pct_municipios":100*xs[:,4].mean(),
      "RI_mediana_kg_mes":d["RI_kg_mes"].median(),
      "CRR_mediana_ton_mes":d["CRR_ton_mes_contexto"].median(),
      "recuperacion_relativa_mediana_pct":100*d["RI_intensidad"].median(),
      "silueta_media_grupo":np.mean(silueta_individual[mask]),
      "ceros_ambiguos":int(d["RI_estado"].eq("CERO AMBIGUO").sum())
    })
    idx=np.flatnonzero(mask)
    dist=np.linalg.norm(X215[idx]-centroide,axis=1)
    orden=idx[np.argsort(dist)]
    validos=[i for i in orden if principal.iloc[i]["RI_estado"]!="CERO AMBIGUO"]
    for r,i in enumerate((validos if validos else list(orden))[:3],start=1):
        filas_representantes.append({"grupo":f"G{g}","orden_cercania":r,
          "CantonId":principal.iloc[i]["CantonId"],
          "municipio":principal.iloc[i]["NombreCantón"],
          "provincia_codigo":principal.iloc[i]["Provin"],
          "distancia_centroide":float(np.linalg.norm(X215[i]-centroide)),
          "silueta_individual":float(silueta_individual[i])})
perfiles=pd.DataFrame(filas_perfiles)
representantes=pd.DataFrame(filas_representantes)
asignaciones=principal[["CantonId","NombreCantón","Provin"]].copy()
asignaciones["grupo_kmeans"]=[f"G{g}" for g in YKM]
asignaciones["grupo_ward_sin_correspondencia"]=[f"W{g}" for g in YWARD]
asignaciones["silueta_5d"] = silueta_individual

mostrar=["grupo","municipios","barrido_medio_pct","diferenciada_media_pct",
 "organicos_pct_municipios","botadero_pct_municipios","RI_mediana_kg_mes","CRR_mediana_ton_mes"]
display(perfiles[mostrar].round(2))
print("Tres municipios cercanos a cada centroide; NO son ejemplos de mejor gestión:")
display(representantes[representantes.orden_cercania==1])

In [ ]:
medidas=["barrido_medio_pct","diferenciada_media_pct","organicos_pct_municipios",
         "recuperacion_relativa_mediana_pct","botadero_pct_municipios"]
matriz_calor=perfiles[medidas].to_numpy(dtype=float)
fig,ax=plt.subplots(figsize=(11,4.5))
img=ax.imshow(np.clip(matriz_calor,0,100),aspect="auto",vmin=0,vmax=100,cmap="YlGnBu")
ax.set(xticks=np.arange(5),yticks=np.arange(5),
  xticklabels=["Barrido","Diferenciada","Orgánicos","Recup. relativa mediana","Botadero"],
  yticklabels=[f"G{i} ({int(perfiles.loc[i,'municipios'])})" for i in range(5)],
  title="Cinco perfiles descriptivos (porcentajes, no desempeño)")
for i in range(5):
    for j in range(5):
        ax.text(j,i,f"{matriz_calor[i,j]:.1f}",ha="center",va="center",
                color="white" if matriz_calor[i,j]>55 else "black")
fig.colorbar(img,ax=ax,label="Escala visual limitada a 100 %")
plt.tight_layout()
fig.savefig(CARPETA_SALIDA/"04_perfiles.png",dpi=160,bbox_inches="tight")
plt.show()

### Lectura en lenguaje municipal

- **G0:** casi sin recolección diferenciada ni actividad orgánica declarada; revisar registros, rutas y posibilidades de separación en origen.
- **G1:** diferenciación alta junto con actividad orgánica declarada; comprobar continuidad, destino y cantidad realmente aprovechada.
- **G2:** declaración de botadero como disposición principal; verificar condiciones del sitio y alternativas técnicas.
- **G3:** diferenciación alta sin actividad orgánica declarada; examinar flujos separados y registro de corrientes orgánicas.
- **G4:** actividad orgánica declarada con diferenciación limitada; investigar su integración con la recolección.

Los grupos describen combinaciones observadas y **no** una escala de mejor a peor. Los nombres municipales cercanos al centroide señalan semejanza estadística, no excelencia operativa.

## 6. PCA: visualización y explicación de variabilidad

Se ajusta `PCA(n_components=5, svd_solver="full")` en la **misma matriz principal de cinco variables sin z-score**. Los puntos del gráfico muestran la proyección PC1–PC2; las siluetas y los agrupamientos anteriores **no** se vuelven a calcular sobre esta proyección. Las contribuciones de cada característica a un eje se calculan como el cuadrado del elemento del vector propio correspondiente, expresado en porcentaje.

In [ ]:
pca=PCA(n_components=5,svd_solver="full").fit(X215)
coords=pca.transform(X215)
var_pc=pca.explained_variance_ratio_*100
var_acum=np.cumsum(var_pc)
print("Varianza explicada individual (%)",np.round(var_pc,4))
print("Varianza acumulada (%)",np.round(var_acum,4))
print(f"PC1+PC2: {var_acum[1]:.4f}% | Varianza no vista en 2D: {100-var_acum[1]:.4f}%")

pca_pesos=pd.DataFrame(pca.components_,columns=ATRIBUTOS,index=[f"PC{i}" for i in range(1,6)])
pca_contrib=(pca_pesos**2)*100
pca_resumen=pd.DataFrame({"componente":[f"PC{i}" for i in range(1,6)],
    "varianza_pct":var_pc,"acumulada_pct":var_acum})
display(pca_resumen.round(4))
display(pca_contrib.iloc[:2].round(2))

fig,ax=plt.subplots(figsize=(8,4))
ax.bar(np.arange(1,6),var_pc,label="Individual")
ax.plot(np.arange(1,6),var_acum,"o-",color="black",label="Acumulada")
ax.axhline(90,color="gray",linestyle="--",alpha=.8)
ax.set(xlabel="Componente principal",ylabel="Varianza explicada (%)",xticks=list(range(1,6)),
       title="Varianza individual y acumulada del PCA")
ax.legend();ax.grid(axis="y",alpha=.2);plt.tight_layout()
fig.savefig(CARPETA_SALIDA/"05_pca_varianza.png",dpi=160,bbox_inches="tight")
plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(8,6))
for g in range(5):
    mask=YKM==g
    ax.scatter(coords[mask,0],coords[mask,1],s=34,alpha=.75,label=f"G{g}: {mask.sum()}")
ax.set(xlabel=f"PC1 ({var_pc[0]:.2f}% de varianza)",ylabel=f"PC2 ({var_pc[1]:.2f}% de varianza)",
       title="Municipios proyectados en PCA · coloreados por K-Means k=5")
ax.legend(title="Etiquetas descriptivas, no orden de calidad")
ax.grid(alpha=.2);plt.tight_layout()
fig.savefig(CARPETA_SALIDA/"06_pca_proyeccion.png",dpi=160,bbox_inches="tight")
plt.show()

fig,ax=plt.subplots(figsize=(10,4))
indices=np.arange(5);ancho=.35
ax.bar(indices-ancho/2,pca_contrib.iloc[0].values,width=ancho,label="PC1")
ax.bar(indices+ancho/2,pca_contrib.iloc[1].values,width=ancho,label="PC2")
ax.set(xticks=indices,xticklabels=["Barrido","Diferenciada","Orgánicos","Recup. RI","Botadero"],
       ylabel="Contribución al eje (%)",title="Aportes de las características a los dos primeros ejes")
ax.legend();ax.grid(axis="y",alpha=.2);plt.tight_layout()
fig.savefig(CARPETA_SALIDA/"07_pca_contribuciones.png",dpi=160,bbox_inches="tight")
plt.show()

## 7. Estabilidad y sensibilidad de las conclusiones

Se comparan las asignaciones sobre **los mismos 204 municipios compartidos**, evitando contrastar directamente vectores de distinta longitud. Para reportar cuántos cambiaron, se alinean las etiquetas mediante asignación óptima (algoritmo húngaro). Después se contrastan tres decisiones metodológicas: z-score, recuperación **absoluta** en vez de relativa, y una sexta variable de volumen `log1p(CRR)` estandarizada. Las tres son *sensibilidades*, no modificaciones al modelo principal.

In [ ]:
ids_comunes=sorted(set(principal["CantonId"]) & set(conservadora["CantonId"]))
assert len(ids_comunes)==204
estabilidad=[]
for metodo in ("K-Means","Ward"):
    for k in VALORES_K:
        ya=dict(zip(principal["CantonId"],etiquetas_guardadas[("215",metodo,k)]))
        yb=dict(zip(conservadora["CantonId"],etiquetas_guardadas[("204",metodo,k)]))
        a=np.array([ya[id_] for id_ in ids_comunes])
        b=np.array([yb[id_] for id_ in ids_comunes])
        coincidencias=alinear_y_contar(a,b)
        estabilidad.append({"metodo":metodo,"k":k,"municipios_comunes":len(ids_comunes),
            "ARI_215_vs_204":adjusted_rand_score(a,b),
            "cambios_tras_alinear":len(ids_comunes)-coincidencias})
estabilidad_df=pd.DataFrame(estabilidad)
display(estabilidad_df.round(6))
assert int(estabilidad_df.query('metodo=="K-Means" and k==5').iloc[0].cambios_tras_alinear)==25

fig,ax=plt.subplots(figsize=(9,4))
for n in (215,204):
    d=comparacion[(comparacion.escenario==n)&(comparacion.metodo=="K-Means")].sort_values("k")
    ax.plot(d.k,d.silueta_5d,"o-",label=f"{n} municipios")
ax.set(xticks=list(VALORES_K),xlabel="Número de grupos k",ylabel="Silueta en cinco dimensiones",
       title="Efecto de retirar once ceros ambiguos de RI")
ax.legend();ax.grid(alpha=.2);plt.tight_layout()
fig.savefig(CARPETA_SALIDA/"08_sensibilidad_215_204.png",dpi=160,bbox_inches="tight")
plt.show()

In [ ]:
matriz_z=StandardScaler().fit_transform(X215)
q95_ri_absoluto=np.percentile(conservadora["RI_kg_mes"].to_numpy(dtype=float),95)
assert q95_ri_absoluto > 0
matriz_absoluta=X215.copy()
matriz_absoluta[:,3]=np.log1p(principal["RI_kg_mes"].to_numpy(dtype=float))/np.log1p(q95_ri_absoluto)
ln_volumen=np.log1p(principal["CRR_ton_mes_contexto"].to_numpy(dtype=float))
assert np.isfinite(ln_volumen).all() and np.std(ln_volumen)>0
matriz_6=np.column_stack([X215,(ln_volumen-np.mean(ln_volumen))/np.std(ln_volumen)])
experimentos=[("Z-score",matriz_z),("RI absoluta",matriz_absoluta),("Agregar volumen log estandarizado",matriz_6)]
filas_sens=[]
for metodo in ("K-Means","Ward"):
    baseline=etiquetas_guardadas[("215",metodo,5)]
    for descripcion,mat in experimentos:
        y_alt=kmeans(mat,5,SEMILLA_ELEGIDA) if metodo=="K-Means" else ward(mat,5)
        filas_sens.append({"metodo":metodo,"tratamiento":descripcion,"ARI_vs_principal":adjusted_rand_score(baseline,y_alt),
             "silueta_matriz_alternativa":silueta(mat,y_alt),"coincidencias_alineadas":alinear_y_contar(baseline,y_alt)})
sensibilidad=pd.DataFrame(filas_sens)
display(sensibilidad.round(6))

## 8. Control de concordancia con el informe PUCE

Los valores del informe sirven como **referencias independientes**. El notebook no los usa para ajustar modelos ni para sustituir resultados. Si no coinciden, se detiene la ejecución con una explicación: habría que revisar el CSV, la versión de dependencias, el método o el informe. El contraste se hace con tolerancias pequeñas, porque los resultados se reportaron redondeados.
Se incluye una comprobación adicional de **ARI medio y mínimo entre semillas**, porque la solución de la semilla 52 detecta pequeñas variaciones en la lectura decimal. El control de integridad SHA256 indica si la entrada coincide con el CSV preparado que figura en el manifiesto del Hito 2.


In [ ]:
registro_semillas_k5 = comparacion.query('escenario==215 and metodo=="K-Means" and k==5').iloc[0]
referencias={
  "ARI_semillas_medio_k5":(registro_semillas_k5.ARI_semillas_medio,0.994993,0.0001),
  "ARI_semillas_minimo_k5":(registro_semillas_k5.ARI_semillas_minimo,0.987483,0.0001),
  "silueta_kmeans":(silueta_km,0.593507,0.0001),
  "silueta_ward":(silueta_ward,0.578667,0.0001),
  "ari_metodos":(ari_metodos,0.879070,0.0001),
  "PCA_PC1_pct":(var_pc[0],46.6633,0.01),
  "PCA_PC2_pct":(var_pc[1],18.0605,0.01),
  "PCA_acumulado_2_pct":(var_acum[1],64.7238,0.01),
  "ARI_escenarios_kmeans_k5":(float(estabilidad_df.query('metodo=="K-Means" and k==5').iloc[0].ARI_215_vs_204),0.841207,0.0001),
  "ARI_zscore_kmeans":(float(sensibilidad.query('metodo=="K-Means" and tratamiento=="Z-score"').iloc[0].ARI_vs_principal),0.828187,0.0001),
  "ARI_RI_absoluta_kmeans":(float(sensibilidad.query('metodo=="K-Means" and tratamiento=="RI absoluta"').iloc[0].ARI_vs_principal),0.684649,0.0001),
  "ARI_volumen_kmeans":(float(sensibilidad.query('metodo=="K-Means" and tratamiento=="Agregar volumen log estandarizado"').iloc[0].ARI_vs_principal),0.340370,0.0001),
}
control_informe=pd.DataFrame([{"prueba":k,"calculado":float(v),"informe":ref,
 "diferencia_absoluta":abs(v-ref),"coincide":bool(np.isclose(v,ref,atol=tol,rtol=0))}
 for k,(v,ref,tol) in referencias.items()])
display(control_informe.round(6))
if not control_informe.coincide.all():
    raise AssertionError("Hay diferencias respecto al PDF. Revisar entrada/versiones y NO entregar cifras discordantes.")
print("VALIDACIÓN FINAL: resultados independientes compatibles con InformeReto01_PUCE_Revisado.pdf")

## 9. Conclusión técnica y condiciones de uso

La solución descriptiva de **cinco perfiles** distingue combinaciones de barrido, diferenciación, actividad orgánica, recuperación inorgánica relativa y disposición final. Aunque **seis** grupos consiguen una silueta superior, cinco ofrece una descripción útil sin reducir el análisis a una sola métrica. La concordancia de K-Means y Ward es elevada, pero la solución **sí cambia** al tratar ciertos registros o añadir tamaño operativo.

El gráfico PCA es un apoyo: dos componentes no contienen toda la variabilidad y **no reemplazan** la evaluación en cinco dimensiones. El trabajo no identifica a los mejores ni peores municipios. Antes de recomendar intervenciones, los equipos técnicos deberían contrastar calidad de registros, instalaciones, composición, rutas, frecuencia y cantidades efectivamente procesadas.

**Limitaciones que deben acompañar la entrega:** una sola edición (2024); extracto preparado de 22 columnas derivado de 41 campos, no base original oficial de 213; posibles ceros iniciales previamente perdidos en `CantonId`; once RI ambiguos; cuatro coberturas superiores a 100%; variables binarias que no representan toneladas; indicadores relativos cuya comparabilidad depende del alcance del reporte.

In [ ]:
# Exportar resultados derivados; ningún CSV de resultados reemplaza los datos municipales originales.
comparacion.to_csv(CARPETA_SALIDA/"comparacion_metodos.csv",index=False)
semillas_df.to_csv(CARPETA_SALIDA/"estabilidad_semillas.csv",index=False)
perfiles.to_csv(CARPETA_SALIDA/"perfiles_municipales.csv",index=False)
representantes.to_csv(CARPETA_SALIDA/"municipios_representativos.csv",index=False)
asignaciones.to_csv(CARPETA_SALIDA/"asignaciones_municipales.csv",index=False)
estabilidad_df.to_csv(CARPETA_SALIDA/"sensibilidad_muestras.csv",index=False)
sensibilidad.to_csv(CARPETA_SALIDA/"sensibilidad_variables.csv",index=False)
pca_resumen.to_csv(CARPETA_SALIDA/"pca_varianza.csv",index=False)
pca_contrib.to_csv(CARPETA_SALIDA/"pca_contribuciones.csv")
control_informe.to_csv(CARPETA_SALIDA/"control_concordancia_informe.csv",index=False)

meta={"edicion":"INEC GIRS 2024","fuente_csv_preparado":str(origen),
  "commit_repositorio":COMMIT_FIJO,"fecha_ejecucion_utc":datetime.now(timezone.utc).isoformat(),
  "dependencias":{"python":sys.version.split()[0],"numpy":np.__version__,"pandas":pd.__version__,
     "sklearn":sklearn.__version__,"scipy":scipy.__version__},
  "muestras":{"auditada":len(base),"principal":len(principal),"conservadora":len(conservadora)},
  "k_propuesto":5,"semillas":list(SEMILLAS),"n_init":N_INICIALIZACIONES,
  "caracteristicas":ATRIBUTOS,"controles":controles,
  "calidad":{"silueta_kmeans_k5":silueta_km,"silueta_ward_k5":silueta_ward,
             "ari_metodos_k5":ari_metodos,"pca_2d_varianza_pct":float(var_acum[1])},
  "validacion_informe":bool(control_informe.coincide.all()),
  "sha256_csv_recibido": sha256_real,
  "coincide_manifesto_csv_preparado":bool(coincide_csv_institucional_preparado),
  "precaucion":"Agrupamientos exploratorios; no son un ranking, ni permiten inferir causalidad."}
(CARPETA_SALIDA/"metadatos_ejecucion.json").write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding="utf-8")
zip_salida=Path("Reto01_GIRS_2024_Resultados_Colab.zip")
with zipfile.ZipFile(zip_salida,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(CARPETA_SALIDA.glob("*")):
        zf.write(p,p.relative_to(CARPETA_SALIDA.parent))
print("Archivos generados:",len(list(CARPETA_SALIDA.glob("*"))))
print("ZIP:",zip_salida.resolve())
# Para descargar desde Colab, descomentar las dos siguientes líneas:
# from google.colab import files
# files.download(str(zip_salida))